In [151]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [152]:
"""
What we are gonna do here is 
Data Inspection & Integrity Check
"""

# Load dataset
df = pd.read_csv('AI-Dataset.csv')

# Preview the first 5 records
print("--- FIRST 5 ROWS ---")
df.head()

--- FIRST 5 ROWS ---


,Age,Education_Level,Years_of_Experience,Industry,Job_Role,Company_Size,Job_Level,Routine_Task_Percentage,Creativity_Requirement,Human_Interaction_Level,AI_Adoption_Level,Number_of_AI_Tools_Used,AI_Usage_Hours_Per_Week,Tasks_Automated_Percentage,AI_Training_Hours,Layoff_Risk
0,59,Master's,6,Finance,Accountant,Medium,Entry,84,21,94,Medium,5,7,40,26,High
1,44,Master's,14,Manufacturing,Production Supervisor,Small,Entry,30,84,68,Low,2,2,14,9,Low
2,36,Bachelor's,7,Retail,Store Manager,Medium,Senior,12,86,71,Low,0,1,6,3,Low
3,27,Bachelor's,6,Finance,Auditor,Large,Entry,90,18,21,Medium,3,10,67,26,High
4,49,High School,12,Finance,Auditor,Small,Entry,49,52,72,Medium,5,13,26,19,Medium


In [153]:
print('---------All Features---------------')
print(*df.columns, sep="\n")
print(f'The number of feactures: {df.shape[1]}')

---------All Features---------------
Age
Education_Level
Years_of_Experience
Industry
Job_Role
Company_Size
Job_Level
Routine_Task_Percentage
Creativity_Requirement
Human_Interaction_Level
AI_Adoption_Level
Number_of_AI_Tools_Used
AI_Usage_Hours_Per_Week
Tasks_Automated_Percentage
AI_Training_Hours
Layoff_Risk
The number of feactures: 16


In [154]:
# Dimensions and column data types
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Age                         20000 non-null  int64
 1   Education_Level             20000 non-null  str  
 2   Years_of_Experience         20000 non-null  int64
 3   Industry                    20000 non-null  str  
 4   Job_Role                    20000 non-null  str  
 5   Company_Size                20000 non-null  str  
 6   Job_Level                   20000 non-null  str  
 7   Routine_Task_Percentage     20000 non-null  int64
 8   Creativity_Requirement      20000 non-null  int64
 9   Human_Interaction_Level     20000 non-null  int64
 10  AI_Adoption_Level           20000 non-null  str  
 11  Number_of_AI_Tools_Used     20000 non-null  int64
 12  AI_Usage_Hours_Per_Week     20000 non-null  int64
 13  Tasks_Automated_Percentage  20000 non-null  int64
 14  AI_Training_Hours

In [155]:
# 3. Duplicate row check
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate Rows: {duplicate_count}")
if duplicate_count == 0:
    print('There is no duplicate row')


Duplicate Rows: 0
There is no duplicate row


In [156]:
# Target: Layoff_Risk risk to predit
target = 'Layoff_Risk'
# Target variable class distribution
# Calculate counts and percentages
risk_table = (
    df[target]
    .value_counts()
    .reindex(["Low", "Medium", "High"])
    .rename_axis("Risk category")
    .reset_index(name="Records")
)

risk_table["Percentage"] = (risk_table["Records"] / len(df)) * 100

# Add the Total row
total_row = pd.DataFrame({
    "Risk category": ["Total"],
    "Records": [risk_table["Records"].sum()],
    "Percentage": [100.0]
})
risk_table_with_total = pd.concat([risk_table, total_row], ignore_index=True)

# Format and display with the caption
display(
    risk_table_with_total.style
    .set_caption("Distribution of Layoff Risk Categories")
    .set_table_styles([
        {
            'selector': 'caption',
            'props': [
                ('font-size', '16px'),
                ('font-weight', 'bold'),
                ('text-align', 'left'),
                ('padding-bottom', '8px')
            ]
        }
    ])
    .format({"Records": "{:,}", "Percentage": "{:.2f}%"})
    .hide(axis="index")
)

Risk category,Records,Percentage
Low,"6,602",33.01%
Medium,"6,601",33.01%
High,"6,797",33.98%
Total,"20,000",100.00%


**MISSING VALUE & ANOMALY AUDIT:**

In [157]:
# -------------------------------------------------------------------------
# Audit Missing / Null Values Across All Columns
# Here we check if any of the 16 features or the target contain NaN, None,
# or missing records that would require imputation or deletion.
# -------------------------------------------------------------------------
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_audit_table = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage (%)': missing_percent
})

print("--- MISSING VALUE AUDIT (ALL COLUMNS) ---")
display(missing_audit_table)

if (missing_counts == 0).all():
    print("No missing value: You're good to go.")
else:
    print("There are missing values.")

--- MISSING VALUE AUDIT (ALL COLUMNS) ---


,Missing Count,Missing Percentage (%)
Age,0,0.0
Education_Level,0,0.0
Years_of_Experience,0,0.0
Industry,0,0.0
Job_Role,0,0.0
Company_Size,0,0.0
Job_Level,0,0.0
Routine_Task_Percentage,0,0.0
Creativity_Requirement,0,0.0
Human_Interaction_Level,0,0.0


No missing value: You're good to go.


In [158]:
# -------------------------------------------------------------------------
# Numeric Boundary & Range Audit
# We must verify that numeric features fall within valid domain ranges
# (e.g., non-negative hours, percentages strictly between 0 and 100).
# Just Min and Max matters here for checking
# -------------------------------------------------------------------------
print("\n--- SUB-STEP 2.2: NUMERIC RANGE AUDIT ---")
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns
numeric_summary = df[numeric_columns].describe().T[['min', 'mean', '50%', 'max', 'std']]
numeric_summary.columns = ['Min', 'Mean', 'Median (50%)', 'Max', 'Std Dev']
display(numeric_summary)


--- SUB-STEP 2.2: NUMERIC RANGE AUDIT ---


,Min,Mean,Median (50%),Max,Std Dev
Age,21.0,40.36150,40.0,60.0,11.526333
Years_of_Experience,0.0,7.26550,7.0,32.0,4.720606
Routine_Task_Percentage,10.0,51.88550,52.0,94.0,24.437808
Creativity_Requirement,0.0,47.78670,48.0,100.0,26.100482
Human_Interaction_Level,20.0,60.05500,62.0,99.0,22.267801
Number_of_AI_Tools_Used,0.0,2.46035,2.0,10.0,2.097486
AI_Usage_Hours_Per_Week,0.0,6.77395,5.0,30.0,6.086370
Tasks_Automated_Percentage,4.0,37.22895,35.0,93.0,19.995120
AI_Training_Hours,0.0,12.62140,8.0,79.0,13.552647


In [159]:
# -------------------------------------------------------------------------
# Categorical Domain & Label Integrity Audit
# We must inspect all unique values in every categorical column to catch
# misspellings, trailing whitespaces, unexpected categories, or bad encodings.
# -------------------------------------------------------------------------
print("\n--- SUB-STEP 2.4: CATEGORICAL VALUES AUDIT ---")
categorical_columns = df.select_dtypes(include=['object', 'string']).columns

for col in categorical_columns:
    unique_vals = df[col].unique()
    print(f"Column '{col}' : {unique_vals.tolist()}")


--- SUB-STEP 2.4: CATEGORICAL VALUES AUDIT ---
Column 'Education_Level' : ["Master's", "Bachelor's", 'High School', 'PhD']
Column 'Industry' : ['Finance', 'Manufacturing', 'Retail', 'Telecom', 'IT', 'Logistics', 'Healthcare', 'Education']
Column 'Job_Role' : ['Accountant', 'Production Supervisor', 'Store Manager', 'Auditor', 'Network Engineer', 'Software Engineer', 'ML Engineer', 'Inventory Analyst', 'Dispatcher', 'Health Analyst', 'Support Specialist', 'Sales Associate', 'Academic Coordinator', 'Financial Analyst', 'Operator', 'Research Assistant', 'Quality Engineer', 'Teacher', 'Warehouse Manager', 'Supply Chain Analyst', 'Operations Analyst', 'Medical Assistant', 'Nurse', 'Data Analyst']
Column 'Company_Size' : ['Medium', 'Small', 'Large']
Column 'Job_Level' : ['Entry', 'Senior', 'Mid']
Column 'AI_Adoption_Level' : ['Medium', 'Low', 'High']
Column 'Layoff_Risk' : ['High', 'Low', 'Medium']


**FEATURE CATEGORIZATION & ENCODING STRATEGY:**

Here we choose specific features we selected to work with to predict the layoff risk.

We seperate features from the target, and trasform all categorical data into numerical format (Encoding)

that is suitable for model we train.

**Selected Features:**

Age, Years_of_Experience, Industry, Job_Level, Education_Level, AI_Adoption_Level


## Strategy Breakdown for our selected features:

### Continuous Numeric Features (No encoding needed, ready for scaling):
    Age, Years_of_Experience

### Categorical Features:
    Education_Level:
        'High School': 0 , "Bachelor's": 1 , "Master's": 2 , 'PhD': 3
    
### Nominal Categorical Feature:
    Industry: Converted to binary dummy columns (drop_first=True avoids multicollinearity / the dummy variable trap).
        For each kind of job we have the column. If that person is in the company, we show 1, otherwise 0

### Target Variable:
    Layoff_Risk: 'Low': 0, 'Medium': 1, 'High': 2



In [176]:
# -------------------------------------------------------------------------
# We must extract strictly the 6 chosen predictor features and the target
# variable from the main DataFrame.
# -------------------------------------------------------------------------

selected_features = [
    "Age",
    "Years_of_Experience",
    "Industry",
    "Job_Level",
    "Education_Level",
    "AI_Adoption_Level"
]

target = "Layoff_Risk"

# Creating an explicit working copy of the subs
df_model = df[selected_features + [target]].copy()
print("--- Dataframe Model Row and column check---")
print(f"Rows: {df_model.shape[0]}, Selected Columns: {df_model.shape[1]}")

--- Dataframe Model Row and column check---
Rows: 20000, Selected Columns: 7


In [177]:
# -------------------------------------------------------------------------
# Target Variable Encoding
# We convert the multi-class text labels of 'Layoff_Risk' into
# standard integer labels: Low -> 0, Medium -> 1, High -> 2.
# -------------------------------------------------------------------------
target_mapping = {
    'Low': 0,
    'Medium': 1,
    'High': 2
}

df_model[target] = df_model[target].map(target_mapping)

print("\n--- TARGET ENCODING VERIFICATION ---")
print(df_model['Layoff_Risk'].value_counts().sort_index())


--- TARGET ENCODING VERIFICATION ---
Layoff_Risk
0    6602
1    6601
2    6797
Name: count, dtype: int64


In [178]:
# -------------------------------------------------------------------------
# Ordinal Feature Encoding
# We map ranked categorical variables to integers reflecting their
# intrinsic order as we explained above how we do it.
# -------------------------------------------------------------------------
education_order = {
    'High School': 0,
    "Bachelor's": 1,
    "Master's": 2,
    'PhD': 3
}

job_level_order = {
    'Entry': 0,
    'Mid': 1,
    'Senior': 2
}

ai_adoption_order = {
    'Low': 0,
    'Medium': 1,
    'High': 2
}

df_model['Education_Level'] = df_model['Education_Level'].map(education_order)
df_model['Job_Level'] = df_model['Job_Level'].map(job_level_order)
df_model['AI_Adoption_Level'] = df_model['AI_Adoption_Level'].map(ai_adoption_order)

print("\n--- SUB-STEP 3.3: ORDINAL FEATURES ENCODED SAMPLE ---")
display(df_model[['Education_Level', 'Job_Level', 'AI_Adoption_Level']].head())



--- SUB-STEP 3.3: ORDINAL FEATURES ENCODED SAMPLE ---


,Education_Level,Job_Level,AI_Adoption_Level
0,2,0,1
1,2,0,0
2,1,2,0
3,1,0,1
4,0,0,1


In [ ]:
# -------------------------------------------------------------------------
# Nominal Feature One-Hot Encoding
# We convert unordered categorical column ('Industry') into binary
# dummy indicators. Using drop_first=True prevents multicollinearity.
# -------------------------------------------------------------------------

# The model with encoded value
df_encoded = pd.get_dummies(
    df_model,
    columns=['Industry'],
    drop_first=True,
    dtype=int
)

print("\n--- ORDINAL FEATURES ENCODED SAMPLE ---")
print(f"Transformed Shape: {df_encoded.shape}")
display(df_encoded.head())


--- ORDINAL FEATURES ENCODED SAMPLE ---
Transformed Shape: (20000, 13)


,Age,Years_of_Experience,Job_Level,Education_Level,AI_Adoption_Level,Layoff_Risk,Industry_Finance,Industry_Healthcare,Industry_IT,Industry_Logistics,Industry_Manufacturing,Industry_Retail,Industry_Telecom
0,59,6,0,2,1,2,1,0,0,0,0,0,0
1,44,14,0,2,0,0,0,0,0,0,1,0,0
2,36,7,2,1,0,0,0,0,0,0,0,1,0
3,27,6,0,1,1,2,1,0,0,0,0,0,0
4,49,12,0,0,1,1,1,0,0,0,0,0,0


## Train / Validation / Test Splitting

We decided to partition the prepared dataset ($N = 20,000$) into three distinct sets: Training (70%), Validation (20%), and Testing (10%). 

### Separation of Target ($y$) and Predictors ($X$): 

The response variable Layoff_Risk is isolated from the input features.

### Splitting Logic for Train, Validation, and Test:

1- train_test_split from scikit-learn splits data into two sets at a time.

2- Split 1: We reserve 70% of data for the training set and place the remaining 30%  into a temporary partition.

3- Split 2: We split the temporary 30% partition into 20 % of it for validation and 10 % for testing

In [179]:
# -------------------------------------------------------------------------
# We separate Target Variable (y) from Predictor Features (X)
# We Isolate the feature matrix X and the ground-truth label vector y.
# -------------------------------------------------------------------------

# input features 
X = df_encoded.drop(columns=['Layoff_Risk'])
# Target 
y = df_encoded['Layoff_Risk']

# The reason we have 12 features, is because of industrial categories
print("--- FEATURE TARGET SHAPES ---")
print(f"X shape: {X.shape}, y shape: {y.shape}")

# -------------------------------------------------------------------------
# Split (75% Training, 25% Temporary Holdout)
# We partition out 75% (15,000 records) strictly for model training,
# keeping 25% (5,000 records) for validation and final testing.
# stratify=y guarantees identical class proportions in both sets.
# -------------------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# -------------------------------------------------------------------------
# Second Split (15% Validation, 10% Testing)
# We subdivide the 25% temporary holdout (5,000 rows):
# 60% of 25% = 15% of total data (3,000 rows) for Validation.
# 40% of 25% = 10% of total data (2,000 rows) for Testing.
# -------------------------------------------------------------------------
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.40,
    random_state=42,
    stratify=y_temp
)


# -------------------------------------------------------------------------
# Verification of Split Dimensions and Class Balance
# We print summary tables showing counts and percentages across all
# partitions to confirm no data leakage or class skew was introduced.
# -------------------------------------------------------------------------
split_summary = pd.DataFrame({
    'Subset': ['Training (75%)', 'Validation (15%)', 'Testing (10%)', 'Total'],
    'Row Count': [len(X_train), len(X_val), len(X_test), len(df_encoded)],
    'Percentage': [
        f"{(len(X_train)/len(df_encoded))*100:.1f}%",
        f"{(len(X_val)/len(df_encoded))*100:.1f}%",
        f"{(len(X_test)/len(df_encoded))*100:.1f}%",
        "100.0%"
    ]
})

print("\n--- PARTITION SIZES ---")
display(split_summary)

--- FEATURE TARGET SHAPES ---
X shape: (20000, 12), y shape: (20000,)

--- PARTITION SIZES ---


,Subset,Row Count,Percentage
0,Training (75%),15000,75.0%
1,Validation (15%),3000,15.0%
2,Testing (10%),2000,10.0%
3,Total,20000,100.0%


In [144]:
# Checking class stratification across splits
class_dist = pd.DataFrame({
    'Train (%)': (y_train.value_counts(normalize=True) * 100).round(2),
    'Val (%)': (y_val.value_counts(normalize=True) * 100).round(2),
    'Test (%)': (y_test.value_counts(normalize=True) * 100).round(2)
}, index=[0, 1, 2]).rename(index={0: 'Low (0)', 1: 'Medium (1)', 2: 'High (2)'})

print("\n--- STRATIFICATION CHECK: CLASS DISTRIBUTIONS (%) ---")
display(class_dist)


--- STRATIFICATION CHECK: CLASS DISTRIBUTIONS (%) ---


,Train (%),Val (%),Test (%)
Low (0),33.01,33.03,33.0
Medium (1),33.01,33.00,33.0
High (2),33.99,33.97,34.0
